<a href="https://colab.research.google.com/github/adi-devv/HateScan/blob/main/hate_speech_distilBERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -------------------------------------------------
# 1. Install ONLY the packages you really need
# -------------------------------------------------
!pip install -q --no-cache-dir \
    "torch==2.4.1" "torchvision==0.19.1" "torchaudio==2.4.1" \
    "transformers==4.44.2" "datasets==2.21.0" "huggingface_hub==0.25.1" \
    "fsspec==2023.6.0" "scikit-learn" "pandas" "numpy"

# -------------------------------------------------
# 2. **RESTART RUNTIME** (Runtime → Restart runtime)
# -------------------------------------------------

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 159.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.0/797.0 MB 185.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 154.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 143.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 102.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 264.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.4/436.4 kB 274.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 175.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 162.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Colab_Projects/hateD/dataset/train.csv")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments

# ----------------------------
# 1️⃣ Load your dataset
# ----------------------------
label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

# Create a column to indicate if it's hate (any label is 1)
df['is_hate'] = df[label_cols].sum(axis=1) > 0

# Separate hate and non-hate samples
hate_df = df[df['is_hate']]
non_hate_df = df[~df['is_hate']]

# Downsample non-hate to match hate entries (or some ratio, e.g., 1:1)
non_hate_downsampled = non_hate_df.sample(n=len(hate_df)*1, random_state=42)

# Combine
df_balanced = pd.concat([hate_df, non_hate_downsampled]).sample(frac=1, random_state=42)  # shuffle

# Optional: check
print("Balanced dataset shape:", df_balanced.shape)
print(df_balanced['is_hate'].value_counts())

train_df, val_df = train_test_split(df_balanced, test_size=0.2, random_state=42)

train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

# ----------------------------
# 3️⃣ Tokenization
# ----------------------------
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize(batch):
    return tokenizer(batch['comment_text'], padding=True, truncation=True, max_length=128)

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

# ----------------------------
# 4️⃣ Format Labels
# ----------------------------
def format_labels(batch):
    batch_labels = []
    for i in range(len(batch['toxic'])):
        batch_labels.append([batch[col][i] for col in label_cols])
    batch['labels'] = np.array(batch_labels, dtype=np.float32)
    return batch

train_ds = train_ds.map(format_labels, batched=True)
val_ds = val_ds.map(format_labels, batched=True)

train_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
val_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# ----------------------------
# 5️⃣ Load DistilBERT Model
# ----------------------------
num_labels = len(label_cols)

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# ----------------------------
# 6️⃣ Training Arguments
# ----------------------------
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    logging_dir='./logs',
    logging_steps=50,
    fp16=True if device=="cuda" else False,
    save_strategy="epoch",
    do_eval=True,
    report_to=[]  # disables WandB logging
)

# ----------------------------
# 7️⃣ Trainer Setup
# ----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer
)

# ----------------------------
# 8️⃣ Train & Evaluate
# ----------------------------
trainer.train()
results = trainer.evaluate()
print(results)

# ----------------------------
# 9️⃣ Prediction Function
# ----------------------------
def predict(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.sigmoid(outputs.logits)
    return dict(zip(label_cols, probs.cpu().numpy()[0]))

# Test
print(predict("People like you don't belong here."))

model.save_pretrained('/content/drive/MyDrive/my_distilbert_model')
tokenizer.save_pretrained('/content/drive/MyDrive/my_distilbert_model')


Balanced dataset shape: (32450, 9)
is_hate
True     16225
False    16225
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:90: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
50,0.369800
100,0.226900
150,0.190700
200,0.185700
250,0.184900
300,0.182200
350,0.184400
400,0.177400
450,0.174700
500,0.173300


{'eval_loss': 0.2260206639766693, 'eval_runtime': 6.8092, 'eval_samples_per_second': 953.123, 'eval_steps_per_second': 29.813, 'epoch': 5.0}
{'toxic': 0.9953544, 'severe_toxic': 0.00036829791, 'obscene': 0.00016865274, 'threat': 0.0017546144, 'insult': 0.004433765, 'identity_hate': 0.0004840624}


('/content/drive/MyDrive/my_distilbert_model/tokenizer_config.json',
 '/content/drive/MyDrive/my_distilbert_model/special_tokens_map.json',
 '/content/drive/MyDrive/my_distilbert_model/vocab.txt',
 '/content/drive/MyDrive/my_distilbert_model/added_tokens.json',
 '/content/drive/MyDrive/my_distilbert_model/tokenizer.json')

In [ ]:
# 1️⃣ Load fine-tuned model & tokenizer
# ----------------------------
# from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast
# import torch

# device = "cuda" if torch.cuda.is_available() else "cpu"

# # path to your saved checkpoint
# model_path = "./results/checkpoint-7302"
# model = DistilBertForSequenceClassification.from_pretrained(model_path)
# tokenizer = DistilBertTokenizerFast.from_pretrained(model_path)

# model.to(device)
# model.eval()

# ----------------------------
# 2️⃣ Prediction function (combine all labels)
# ----------------------------
def predict_hate(texts, threshold=0.0):
    """
    texts: list of strings
    threshold: sigmoid probability threshold for considering a label active
    returns: list of dicts: {all_probs_per_label, hate_label}
    """
    inputs = tokenizer(texts, return_tensors="pt", truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()  # shape: (batch_size, num_labels)

    results = []
    for prob in probs:
        # 1 if any class probability > threshold
        hate_label = int((prob >= threshold).any())
        results.append({"probabilities_per_label": prob, "hate_label": hate_label})
    return results

# ----------------------------
# 3️⃣ Test on 4 sentences
# ----------------------------
test_sentences = [
    "People like you don't belong here.",
    "I love everyone equally!",
    "You are a disgrace to humanity.",
    "Yo wassup fu*k!",
]


predictions = predict_hate(test_sentences, threshold=0.5)

for text, pred in zip(test_sentences, predictions):
    # round each probability to 2 decimals
    rounded_probs = [f"{p:.2f}" for p in pred['probabilities_per_label']]

    print(f"Text: {text}")
    print(f"Probabilities per label: {rounded_probs}")
    print(f"Hate label (1=hate, 0=not hate): {pred['hate_label']}")
    print("-" * 60)

Text: People like you don't belong here.
Probabilities per label: ['1.00', '0.00', '0.00', '0.00', '0.00', '0.00']
Hate label (1=hate, 0=not hate): 1
------------------------------------------------------------
Text: I love everyone equally!
Probabilities per label: ['0.00', '0.00', '0.00', '0.00', '0.00', '0.00']
Hate label (1=hate, 0=not hate): 0
------------------------------------------------------------
Text: You are a disgrace to humanity.
Probabilities per label: ['0.99', '0.00', '0.00', '0.01', '0.56', '0.00']
Hate label (1=hate, 0=not hate): 1
------------------------------------------------------------
Text: Yo wassup fu*k!
Probabilities per label: ['1.00', '0.12', '1.00', '0.00', '0.82', '0.00']
Hate label (1=hate, 0=not hate): 1
------------------------------------------------------------


In [ ]:
!pip install optimum[onnxruntime-gpu] onnx onnxruntime-gpu


  Using cached optimum-2.0.0-py3-none-any.whl.metadata (14 kB)
  Using cached transformers-4.57.1-py3-none-any.whl.metadata (43 kB)
  Using cached huggingface_hub-1.0.0-py3-none-any.whl.metadata (13 kB)
  Using cached optimum_onnx-0.0.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 MB 4.4 MB/s eta 0:00:00
Using cached huggingface_hub-0.36.0-py3-none-any.whl (566 kB)
Using cached optimum-2.0.0-py3-none-any.whl (162 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.8/485.8 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 79.7 MB/s eta 0:00:00
Using cached optimum_onnx-0.0.3-py3-none-any.whl (192 kB)


In [ ]:
# -------------------------------------------------
# 3. EXPORT + QUANTIZE → SAVE TO DRIVE
# -------------------------------------------------
from optimum.onnxruntime import ORTModelForSequenceClassification, ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig
from transformers import AutoTokenizer
from pathlib import Path
import os

# ---- INPUT / OUTPUT PATHS (CHANGE ONLY IF NEEDED) ----
INPUT_MODEL  = "/content/drive/MyDrive/my_distilbert_model"          # ← your saved model
OUTPUT_ONNX  = "/content/drive/MyDrive/my_distilbert_onnx_quantized" # ← final quantized model

# Make sure output folder is clean
if os.path.exists(OUTPUT_ONNX):
    import shutil
    shutil.rmtree(OUTPUT_ONNX)
os.makedirs(OUTPUT_ONNX, exist_ok=True)

# ---- 1. Export to ONNX ----
print("Exporting to ONNX...")
ort_model = ORTModelForSequenceClassification.from_pretrained(
    INPUT_MODEL,
    export=True,
    provider="CPUExecutionProvider"   # use "CUDAExecutionProvider" if you have GPU
)
tokenizer = AutoTokenizer.from_pretrained(INPUT_MODEL)

# ---- 2. Quantize (int8) ----
print("Quantizing (int8 → ~66 MB)...")
quantizer = ORTQuantizer.from_pretrained(ort_model)
qconfig = AutoQuantizationConfig.avx512_vnni(
    is_static=False,
    per_channel=True
)
quantizer.quantize(save_dir=OUTPUT_ONNX, quantization_config=qconfig)

# ---- 3. Save tokenizer files ----
tokenizer.save_pretrained(OUTPUT_ONNX)

# ---- 4. Final report ----
total_mb = sum(p.stat().st_size for p in Path(OUTPUT_ONNX).rglob('*') if p.is_file()) / 1e6
print(f"\nQuantized model saved!")
print(f"Folder: {OUTPUT_ONNX}")
print(f"Total size: {total_mb:.1f} MB (~{total_mb:.0f} MB)")

print("\nFiles to use in Chrome extension:")
for f in sorted(Path(OUTPUT_ONNX).rglob("*")):
    if f.is_file():
        print(f"  • {f.name:<25} {f.stat().st_size/1e6:6.2f} MB")

Multiple distributions found for package optimum. Picked distribution: optimum-onnx


Exporting to ONNX...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


Quantizing (int8 → ~66 MB)...

Quantized model saved!
Folder: /content/drive/MyDrive/my_distilbert_onnx_quantized
Total size: 68.5 MB (~69 MB)

Files to use in Chrome extension:
  • config.json                 0.00 MB
  • model_quantized.onnx       67.58 MB
  • ort_config.json             0.00 MB
  • special_tokens_map.json     0.00 MB
  • tokenizer.json              0.71 MB
  • tokenizer_config.json       0.00 MB
  • vocab.txt                   0.23 MB
